# 03 · Table data: load, clean, split and train

Use this notebook when your data is a **table** (a CSV file: one row
per example, one column per measurement) instead of images. It does
for a table what notebooks 01 and 02 do for images:

| Step | What happens |
|---|---|
| 1–3 | Setup, settings, read the CSV |
| 4 | Look at the columns and the missing values |
| 5 | Choose the target and the feature columns |
| 6 | Remove duplicate rows, count the classes |
| 7 | Split into train / validation / test, and save the split |
| 8 | Fill missing values and scale, using the **training** rows only |
| 9–11 | Baseline, train a small network, learning curves, log |
| 12 | Test set, once |
| 13 | Save the files the Streamlit app needs |

**Before you start:** put your CSV file in the `csc451_project`
folder of your Google Drive.

## Step 1 · Setup

In [ ]:
# Run this cell first, every time you open the notebook.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# CHANGE: the address of YOUR team's copy of this repo on GitHub
REPO_URL = "https://github.com/dsc-scsu/csc451-project-starter.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content") / Path(REPO_URL).stem
    if not REPO_DIR.exists():
        !git clone -q {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    !git pull -q
elif Path.cwd().name == "notebooks":
    os.chdir("..")  # on your own computer: work from the repo folder

sys.path.insert(0, "src")
print("Working in", Path.cwd())

## Step 2 · Settings (CHANGE)

| Setting | What to put |
|---|---|
| `CSV_NAME` | Your CSV file, in `PROJECT_FOLDER` on Drive |
| `TARGET_COLUMN` | The column the model predicts (the label) |
| `DROP_COLUMNS` | Columns that must not be features: IDs, names, dates, and anything only known **after** the answer (that would be leakage) |
| `HIDDEN` | Units in the hidden layer. If you change it, change it in `app/app_tabular.py` too |

In [ ]:
# CHANGE: settings for your project
PROJECT_FOLDER = "csc451_project"
CSV_NAME = "data.csv"
TARGET_COLUMN = "label"
DROP_COLUMNS = ["id"]       # columns that are not features

VAL_SIZE = 0.15
TEST_SIZE = 0.15
HIDDEN = 16
LR = 1e-3
EPOCHS = 200
PATIENCE = 20
BATCH_SIZE = 32
SEED = 0
OVERWRITE_SPLITS = False

RUN_NOTE = "first run: default settings"   # CHANGE every run
EVALUATE_TEST = False                      # True only at the very end

In [ ]:
# Where things are. You should not need to change this cell.
if IN_COLAB:
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    DATA_DIR = Path("/content/data")      # fast disk, erased after
else:                                     # the session ends
    PROJECT = Path("project_files")       # on your own computer
    DATA_DIR = PROJECT / "unzipped"
OUT = PROJECT / "outputs"                 # results to keep, on Drive
OUT.mkdir(parents=True, exist_ok=True)
print("Project folder:", PROJECT)

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import train_utils as tu

## Step 3 · Read the CSV

In [ ]:
df = pd.read_csv(PROJECT / CSV_NAME)
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head()

## Step 4 · Look at the columns
`dtype` says how pandas read each column: numbers are `int64` or `float64`; text is `object`. A column of numbers that reads as `object` usually holds a stray value such as `"?"` or `"n/a"`.

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "distinct values": df.nunique(),
})
summary

## Step 5 · Choose the target and the features
This notebook uses the **number** columns as features. Text columns are listed and left out; if one of them matters (for example a color or a category), ask how to turn it into numbers.

In [ ]:
df = df.dropna(subset=[TARGET_COLUMN])
features = df.drop(columns=[TARGET_COLUMN] + DROP_COLUMNS, errors="ignore")
numeric = features.select_dtypes(include="number")
left_out = [c for c in features.columns if c not in numeric.columns]

feature_names = list(numeric.columns)
print(len(feature_names), "feature columns")
if left_out:
    print("Text columns left out:", left_out)

X_all = numeric.to_numpy(dtype=np.float64)
y_names = df[TARGET_COLUMN].astype(str).to_numpy()

## Step 6 · Duplicates and class counts
Identical rows can land in both training and test sets and inflate the test score, so they are removed. The largest class gives the baseline your model has to beat.

In [ ]:
rows = pd.DataFrame(X_all, columns=feature_names)
rows["_y"] = y_names
dup = rows.duplicated().to_numpy()
print(dup.sum(), "duplicate rows removed")
X_all, y_names = X_all[~dup], y_names[~dup]

class_names = sorted(set(y_names))
y_all = np.array([class_names.index(c) for c in y_names])
print(pd.Series(y_names).value_counts().to_string())

## Step 7 · Split, and save the split
A stratified split keeps the same share of each class in train, validation and test. The row numbers of each split are saved, so later runs use the same split.

In [ ]:
splits_file = OUT / "table_splits.json"
if splits_file.exists() and not OVERWRITE_SPLITS:
    with open(splits_file) as f:
        idx = json.load(f)
    print("Using the split saved earlier")
else:
    all_idx = np.arange(len(y_all))
    hold = VAL_SIZE + TEST_SIZE
    tr, rest = train_test_split(all_idx, test_size=hold,
                                stratify=y_all, random_state=SEED)
    va, te = train_test_split(rest, test_size=TEST_SIZE / hold,
                              stratify=y_all[rest], random_state=SEED)
    idx = {"train": tr.tolist(), "val": va.tolist(), "test": te.tolist()}
    with open(splits_file, "w") as f:
        json.dump(idx, f)
    print("Saved", splits_file)

X_tr, y_tr = X_all[idx["train"]], y_all[idx["train"]]
X_va, y_va = X_all[idx["val"]], y_all[idx["val"]]
X_te, y_te = X_all[idx["test"]], y_all[idx["test"]]
print(len(y_tr), "train,", len(y_va), "val,", len(y_te), "test rows")

## Step 8 · Fill missing values and scale (training rows only)

- A missing value is replaced by the **median** of that column.
- Each column is then scaled: subtract its mean, divide by its
  standard deviation, so every feature has a similar range.

The median, mean and standard deviation come from the **training rows
only**. Computing them on all rows lets the test set leak into
training (Week 13). The same numbers are saved for the app.

In [ ]:
medians = np.nanmedian(X_tr, axis=0)
medians = np.where(np.isnan(medians), 0.0, medians)

def fill_missing(X):
    X = X.copy()
    rows_i, cols_i = np.where(np.isnan(X))
    X[rows_i, cols_i] = medians[cols_i]
    return X

X_tr, X_va, X_te = fill_missing(X_tr), fill_missing(X_va), fill_missing(X_te)
mean = X_tr.mean(axis=0)
std = X_tr.std(axis=0)
std[std == 0] = 1.0          # a constant column: avoid dividing by 0

def to_loader(X, y, shuffle):
    X = torch.tensor((X - mean) / std, dtype=torch.float32)
    ds = TensorDataset(X, torch.tensor(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_dl = to_loader(X_tr, y_tr, shuffle=True)
val_dl = to_loader(X_va, y_va, shuffle=False)
test_dl = to_loader(X_te, y_te, shuffle=False)

## Step 9 · Baseline

In [ ]:
largest = np.bincount(y_tr).argmax()
baseline_val = (y_va == largest).mean()
print(f"Always answering '{class_names[largest]}': {baseline_val:.1%} "
      "validation accuracy")

## Step 10 · Train a small network
One hidden layer of `HIDDEN` units. The best epoch by validation accuracy is saved to `outputs/best_table.pt`.

In [ ]:
tu.set_seed(SEED)
device = "cpu"   # small tables train fast on the CPU
model = nn.Sequential(
    nn.Linear(len(feature_names), HIDDEN),
    nn.ReLU(),
    nn.Linear(HIDDEN, len(class_names)),
)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=LR)
checkpoint = OUT / "best_table.pt"
history = tu.fit(model, train_dl, val_dl, loss_fn, opt, device,
                 EPOCHS, checkpoint, PATIENCE)

## Step 11 · Learning curves and experiment log

In [ ]:
hist = pd.DataFrame(history)
best_val = hist["val_acc"].max()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist["epoch"], hist["train_loss"], label="train")
ax1.plot(hist["epoch"], hist["val_loss"], label="validation")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax1.legend()
ax2.plot(hist["epoch"], hist["train_acc"], label="train")
ax2.plot(hist["epoch"], hist["val_acc"], label="validation")
ax2.axhline(baseline_val, color="gray", linestyle="--", label="baseline")
ax2.set_xlabel("epoch")
ax2.set_ylabel("accuracy")
ax2.legend()
plt.tight_layout()
plt.savefig(OUT / "learning_curves_table.png", dpi=150)
plt.show()

tu.log_run(OUT / "experiments_table.csv", note=RUN_NOTE,
           hidden=HIDDEN, lr=LR, epochs_run=len(hist), seed=SEED,
           baseline_val=round(baseline_val, 4),
           best_val_acc=round(best_val, 4))
pd.read_csv(OUT / "experiments_table.csv")

## Step 12 · Test set (once, at the end)
Set `EVALUATE_TEST = True` only for your final model.

In [ ]:
if not EVALUATE_TEST:
    print("Skipped. Set EVALUATE_TEST = True for your final model.")
else:
    y_true, y_pred, probs = tu.predict_all(model, test_dl, device)
    print(f"Test accuracy: {(y_true == y_pred).mean():.1%}   "
          f"(baseline {(y_te == largest).mean():.1%})")
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        colorbar=False)
    plt.title("Test set: rows are the true class")
    plt.show()

## Step 13 · Files for the Streamlit app

`app/app_tabular.py` needs `best.pt` and `info.json`. `info.json`
holds the feature names, the class names, and the training mean and
standard deviation, so the app scales its input exactly as training
did. Copy both files into the repo's `app/` folder.

In [ ]:
info = {
    "features": feature_names,
    "classes": class_names,
    "mean": mean.tolist(),
    "std": std.tolist(),
}
with open(OUT / "info.json", "w") as f:
    json.dump(info, f)

if IN_COLAB:
    from google.colab import files
    files.download(str(checkpoint))        # rename it to best.pt
    files.download(str(OUT / "info.json"))
else:
    print("Copy into app/ (rename best_table.pt to best.pt):")
    print("  ", checkpoint)
    print("  ", OUT / "info.json")